<a href="https://colab.research.google.com/github/Wasey23/CI-345-Machine-Learning-Project/blob/LogisticRegression/Logistic_Regression_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression Model for Movie Success Prediction

## Project Overview

This notebook develops a Logistic Regression model to predict whether a movie is a commercial success.

A movie is classified as:

- 1 = Hit Movie
- 0 = Flop Movie

where:

revenue > 2 × budget

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.preprocessing import (
    StandardScaler,
    MultiLabelBinarizer
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

## Load Dataset

In [2]:
drive.mount('/content/drive')

tmbd_processed = pd.read_csv(
    "/content/drive/MyDrive/CI345_ML/data/tmdb_movies_data.csv"
)

tmbd_processed.head()

Mounted at /content/drive


,id,imdb_id,popularity,budget,revenue,original_title,cast,homepage,director,tagline,...,overview,runtime,genres,production_companies,release_date,vote_count,vote_average,release_year,budget_adj,revenue_adj
0,135397,tt0369610,32.985763,150000000,1513528810,Jurassic World,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,http://www.jurassicworld.com/,Colin Trevorrow,The park is open.,...,Twenty-two years after the events of Jurassic ...,124,Action|Adventure|Science Fiction|Thriller,Universal Studios|Amblin Entertainment|Legenda...,6/9/2015,5562,6.5,2015,137999939.3,1.392446e+09
1,76341,tt1392190,28.419936,150000000,378436354,Mad Max: Fury Road,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,http://www.madmaxmovie.com/,George Miller,What a Lovely Day.,...,An apocalyptic story set in the furthest reach...,120,Action|Adventure|Science Fiction|Thriller,Village Roadshow Pictures|Kennedy Miller Produ...,5/13/2015,6185,7.1,2015,137999939.3,3.481613e+08
2,262500,tt2908446,13.112507,110000000,295238201,Insurgent,Shailene Woodley|Theo James|Kate Winslet|Ansel...,http://www.thedivergentseries.movie/#insurgent,Robert Schwentke,One Choice Can Destroy You,...,Beatrice Prior must confront her inner demons ...,119,Adventure|Science Fiction|Thriller,Summit Entertainment|Mandeville Films|Red Wago...,3/18/2015,2480,6.3,2015,101199955.5,2.716190e+08
3,140607,tt2488496,11.173104,200000000,2068178225,Star Wars: The Force Awakens,Harrison Ford|Mark Hamill|Carrie Fisher|Adam D...,http://www.starwars.com/films/star-wars-episod...,J.J. Abrams,Every generation has a story.,...,Thirty years after defeating the Galactic Empi...,136,Action|Adventure|Science Fiction|Fantasy,Lucasfilm|Truenorth Productions|Bad Robot,12/15/2015,5292,7.5,2015,183999919.0,1.902723e+09
4,168259,tt2820852,9.335014,190000000,1506249360,Furious 7,Vin Diesel|Paul Walker|Jason Statham|Michelle ...,http://www.furious7.com/,James Wan,Vengeance Hits Home,...,Deckard Shaw seeks revenge against Dominic Tor...,137,Action|Crime|Thriller,Universal Pictures|Original Film|Media Rights ...,4/1/2015,2947,7.3,2015,174799923.1,1.385749e+09


In [3]:
tmbd_processed.drop_duplicates(inplace=True)
tmbd_processed.duplicated().sum()

np.int64(0)

## Feature Engineering


In [4]:
tmbd_processed['profit'] = (
    tmbd_processed['revenue']
    - tmbd_processed['budget']
)

tmbd_processed['roi'] = (
    tmbd_processed['revenue']
    / tmbd_processed['budget']
)

## Encode Genres

In [5]:
genres_encoded = (
    tmbd_processed['genres']
    .str.split('|')
    .apply(lambda x: x if isinstance(x, list) else [])
)

mlb = MultiLabelBinarizer()

genre_df = pd.DataFrame(
    mlb.fit_transform(genres_encoded),
    columns=mlb.classes_
)

tmbd_processed = pd.concat(
    [tmbd_processed, genre_df],
    axis=1
)

## Feature Selection

In [6]:
tmbd_processed['release_date'] = pd.to_datetime(tmbd_processed['release_date'])
tmbd_processed['release_month'] = tmbd_processed['release_date'].dt.month

# Create the 'is_hit' target variable
tmbd_processed['is_hit'] = (tmbd_processed['revenue'] > 2 * tmbd_processed['budget']).astype(int)

features = [
    'popularity',
    'runtime',
    'vote_count',
    'vote_average',
    'budget_adj',
    'revenue_adj',
    'release_month',
    'profit',
    'roi'
]

features += list(mlb.classes_)

X = tmbd_processed[features]

y = tmbd_processed['is_hit']

## Class Distribution

In [7]:
print(y.value_counts())

is_hit
0    7850
1    3016
Name: count, dtype: int64


## Train-Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Feature Scaling

In [9]:
# Handle infinite values by replacing them with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Impute NaN values with the mean of the training set
# Calculate mean only from X_train to avoid data leakage
imputer_means = X_train.mean()
X_train = X_train.fillna(imputer_means)
X_test = X_test.fillna(imputer_means)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

## Baseline Logistic Regression

In [10]:
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_model.fit(
    X_train_scaled,
    y_train
)

baseline_pred = baseline_model.predict(
    X_test_scaled
)

## Baseline Model Performance

In [11]:
baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

baseline_precision = precision_score(
    y_test,
    baseline_pred
)

baseline_recall = recall_score(
    y_test,
    baseline_pred
)

baseline_f1 = f1_score(
    y_test,
    baseline_pred
)

print(
    classification_report(
        y_test,
        baseline_pred
    )
)

              precision    recall  f1-score   support

           0       0.92      0.99      0.95      1571
           1       0.98      0.76      0.86       603

    accuracy                           0.93      2174
   macro avg       0.95      0.88      0.90      2174
weighted avg       0.93      0.93      0.93      2174



## Hyperparameter Tuning

In [12]:
param_grid = {
    'C': [0.1, 1, 10],
    'solver': [
        'lbfgs',
        'liblinear'
    ]
}

grid = GridSearchCV(
    LogisticRegression(
        max_iter=1000, # Revert max_iter as saga solver is removed
        class_weight='balanced',
        fit_intercept=True, # Explicitly set fit_intercept
        random_state=42 # Added for reproducibility
    ),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1 # Utilize all available CPU cores
)

grid.fit(
    X_train_scaled,
    y_train
)

print(
    "Best Parameters:",
    grid.best_params_
)

print(
    "Best Score:",
    grid.best_score_
)

Best Parameters: {'C': 10, 'solver': 'lbfgs'}
Best Score: 0.9555915168896586


## Tuned Logistic Regression

In [13]:
best_model = grid.best_estimator_

best_model.fit(
    X_train_scaled,
    y_train
)

tuned_pred = best_model.predict(
    X_test_scaled
)

## Tuned Model Performance

In [14]:
tuned_accuracy = accuracy_score(
    y_test,
    tuned_pred
)

tuned_precision = precision_score(
    y_test,
    tuned_pred
)

tuned_recall = recall_score(
    y_test,
    tuned_pred
)

tuned_f1 = f1_score(
    y_test,
    tuned_pred
)

print(
    classification_report(
        y_test,
        tuned_pred
    )
)

              precision    recall  f1-score   support

           0       0.94      0.99      0.97      1571
           1       0.98      0.84      0.90       603

    accuracy                           0.95      2174
   macro avg       0.96      0.92      0.94      2174
weighted avg       0.95      0.95      0.95      2174



## Confusion Matrix

In [15]:
cm = confusion_matrix(
    y_test,
    tuned_pred
)

print(cm)

[[1559   12]
 [  96  507]]


## Cross Validation

In [16]:
scores = cross_val_score(
    best_model,
    X_train_scaled,
    y_train,
    cv=5
)

print(scores)

print(
    "Average Score:",
    scores.mean()
)

[0.95457159 0.95514664 0.95742232 0.95224396 0.95857307]
Average Score: 0.9555915168896586


## Model Comparison

In [17]:
comparison = pd.DataFrame({
    'Model': [
        'Baseline Logistic Regression',
        'Tuned Logistic Regression'
    ],

    'Accuracy': [
        baseline_accuracy,
        tuned_accuracy
    ],

    'Precision': [
        baseline_precision,
        tuned_precision
    ],

    'Recall': [
        baseline_recall,
        tuned_recall
    ],

    'F1 Score': [
        baseline_f1,
        tuned_f1
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Baseline Logistic Regression,0.929163,0.980728,0.759536,0.856075
1,Tuned Logistic Regression,0.950322,0.976879,0.840796,0.903743


## Conclusion

The Logistic Regression model was trained to predict movie success.

Feature engineering and genre encoding were used to improve predictive performance. Hyperparameter tuning was performed using GridSearchCV, and the tuned model was compared against the baseline model using accuracy, precision, recall, and F1-score.

Cross-validation was also used to evaluate the model's generalization capability.